# Dataset 2: family_data.csv

In [15]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [16]:
data = pd.read_csv("family_data.csv")

In [17]:
# 檢查缺失值
missing_values = data.isnull().sum()
print("缺失值:\n", missing_values)

缺失值:
 Family    0
Member    0
Income    0
Spend     0
dtype: int64


In [18]:
# 檢查重複值
duplicate_rows = data.duplicated().sum()
print("\n重複行數:", duplicate_rows)


重複行數: 0


In [19]:
# 檢查是否有負數（僅針對數值型欄位）
negative_values = (data.select_dtypes(include=['number']) < 0).sum()
print("\n負數值:\n", negative_values)


負數值:
 Income    0
Spend     0
dtype: int64


In [20]:
# 檢查資料類型
data_types = data.dtypes
print("\n數據類型:\n", data_types)


數據類型:
 Family    object
Member    object
Income     int64
Spend      int64
dtype: object


In [21]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Family  279 non-null    object
 1   Member  279 non-null    object
 2   Income  279 non-null    int64 
 3   Spend   279 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 8.8+ KB


In [22]:
data.describe()

,Income,Spend
count,2.790000e+02,2.790000e+02
mean,9.477808e+05,3.344265e+05
std,1.001295e+06,3.760808e+05
min,0.000000e+00,1.391000e+03
25%,0.000000e+00,1.550700e+04
50%,5.458300e+05,1.744480e+05
75%,1.808509e+06,5.432650e+05
max,2.979034e+06,1.475664e+06


In [23]:
data.head()

,Family,Member,Income,Spend
0,family1,Adult1,2376330,1119433
1,family1,Adult2,130268,37337
2,family1,Adult3,2254489,972327
3,family2,Adult1,2292355,649806
4,family2,Adult2,298167,100723


# I. Part 1: Questions Related to Data Analysis

# Q1. Which family boasts the highest annual income, and which has the lowest? How do you ascertain this?

In [24]:
data_grouped = data.groupby("Family").agg({"Income": "sum", "Spend": "sum"})
data_grouped

,Income,Spend
Family,,
family1,4761087,2129097
family10,1675551,560570
family100,1031646,258414
family11,2358538,591169
family12,3794778,1356017
...,...,...
family95,6284612,2574026
family96,325062,135954
family97,2663794,774694


In [26]:
highest_income_family = data_grouped[data_grouped["Income"] == data_grouped["Income"].max()]
highest_income_family

,Income,Spend
Family,,
family6,7804425,2879221


In [27]:
lowest_income_family = data_grouped[data_grouped["Income"] == data_grouped["Income"].min()]
lowest_income_family

,Income,Spend
Family,,
family94,46790,30029


# Q2. Which families do not possess adequate annual income to cover all members' spending? What is the maximum shortfall? How do you determine this?

In [28]:
families_income_less_than_spend = data_grouped[data_grouped["Income"] < data_grouped["Spend"]]
families_income_less_than_spend

,Income,Spend
Family,,


# Q3. Are there any single-parent families, where only one Adult is present? Are there any childless families? How do you discern this?

1.only one Adult is present

In [29]:
df=data
# 將 "Adult" 篩選出來
adults = df[df["Member"].str.contains("Adult")]

# 計算每個 family 中 Adult 的數量
adult_counts = adults.groupby("Family").size()

# 篩選出只有一個 Adult 的 family
single_adult_families = adult_counts[adult_counts == 1].index
single_adult_families

Index(['family14', 'family15', 'family21', 'family22', 'family25', 'family27',
       'family3', 'family32', 'family33', 'family37', 'family38', 'family42',
       'family45', 'family46', 'family48', 'family49', 'family54', 'family55',
       'family59', 'family60', 'family61', 'family64', 'family66', 'family68',
       'family7', 'family72', 'family74', 'family78', 'family79', 'family80',
       'family82', 'family84', 'family85', 'family87', 'family88', 'family89',
       'family93', 'family94', 'family96', 'family99'],
      dtype='object', name='Family')

In [30]:
result = df[df["Family"].isin(single_adult_families)]
result.iloc[:10]

,Family,Member,Income,Spend
8,family3,Adult1,2301931,807835
20,family7,Adult1,2815513,1295014
21,family7,Child1,0,3612
39,family14,Adult1,998732,248730
40,family14,Child1,0,9006
41,family14,Child2,0,3568
42,family15,Adult1,156330,53268
43,family15,Child1,0,7736
44,family15,Child2,0,1391
66,family21,Adult1,2979034,1475664


In [31]:
result

,Family,Member,Income,Spend
8,family3,Adult1,2301931,807835
20,family7,Adult1,2815513,1295014
21,family7,Child1,0,3612
39,family14,Adult1,998732,248730
40,family14,Child1,0,9006
...,...,...,...,...
265,family94,Child1,0,16485
269,family96,Adult1,325062,117236
270,family96,Child1,0,7530
271,family96,Child2,0,11188


2.childless families

In [46]:
# 將 "no_Child" 篩選出來
no_childs = df[~df["Member"].str.contains("Child")]
no_childs

,Family,Member,Income,Spend
0,family1,Adult1,2376330,1119433
1,family1,Adult2,130268,37337
2,family1,Adult3,2254489,972327
3,family2,Adult1,2292355,649806
4,family2,Adult2,298167,100723
...,...,...,...,...
274,family98,Adult1,1445541,587647
275,family98,Adult2,1573068,444308
276,family99,Adult1,1827150,493578
277,family100,Adult1,751899,172111


# Q4. Do you suspect any errors within this dataset? Examples may include negative figures, missing or duplicate data, etc. Why?

想法一:將df中缺失值、重複值、負數值分別取sum()，可以知道出現錯誤的資料數量

In [32]:
# 檢查缺失值
missing_values = df.isnull().sum()

# 檢查重複值
duplicate_rows = df.duplicated().sum()

# 檢查是否有負數（僅針對數值型欄位）
negative_values = (df.select_dtypes(include=['number']) < 0).sum()

# 檢查資料類型
data_types = df.dtypes

# 顯示檢查結果
print("缺失值:\n", missing_values)
print("\n重複行數:", duplicate_rows)
print("\n負數值:\n", negative_values)
print("\n數據類型:\n", data_types)

缺失值:
 Family    0
Member    0
Income    0
Spend     0
dtype: int64

重複行數: 0

負數值:
 Income    0
Spend     0
dtype: int64

數據類型:
 Family    object
Member    object
Income     int64
Spend      int64
dtype: object


想法二:將df中缺失值、重複值、負數值分別存入新建立的df中，相較於想法一，能一目瞭然看到哪筆資料錯誤

In [47]:
# 檢查負數值
negative_values = df[df['Income'] < 0]
#negative_values = df[(df.Income < 0).any(axis=1)]
print("Negative values detected:")
negative_values
# 檢查遺漏值
missing_values = df[df.isnull().any(axis=1)]
print("\nRows with missing values:")
missing_values
# 檢查重複行
duplicate_rows = df[df.duplicated()]
print("\nDuplicate rows detected:")
duplicate_rows
# 額外檢查：檢查欄位中每個 'Family' 是否所有人總收入 (Income) > 0 且合理
grouped = df.groupby("Family")["Income"].sum()
families_with_zero_income = grouped[grouped == 0]
print("\nFamilies with zero total income:")
families_with_zero_income

Negative values detected:

Rows with missing values:

Duplicate rows detected:

Families with zero total income:


Series([], Name: Income, dtype: int64)